In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (df_metric, df_trace, df_log, df_error):
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to build summary and top (cmdb_id, name) pairs with count and global P95
def summarize_kpi(df, name_col):
    total_rows = len(df)
    min_ts = df['ts'].min()
    max_ts = df['ts'].max()
    summary_df = pd.DataFrame([{
        'total_rows': total_rows,
        'min_ts': min_ts,
        'max_ts': max_ts
    }])
    # Group by cmdb_id and KPI name, compute count and P95 across entire series for that pair
    grouped = df.groupby(['cmdb_id', name_col])['value'].agg(
        count='count',
        p95=lambda s: float(np.nanpercentile(s.dropna().values, 95)) if len(s.dropna())>0 else np.nan
    ).reset_index()
    top_pairs = grouped.sort_values('count', ascending=False).head(20)
    return summary_df, top_pairs

# Summaries for metric, trace, log
metric_summary, metric_top_pairs = summarize_kpi(df_metric, 'kpi_name')
trace_summary, trace_top_pairs = summarize_kpi(df_trace, 'trace_name')
log_summary, log_top_pairs = summarize_kpi(df_log, 'log_name')

# Error logs summary: per-cmdb_id counts, min/max timestamp, up to 5 distinct sample messages (truncated)
def truncate(msg, n=200):
    if pd.isna(msg):
        return msg
    s = str(msg)
    return s if len(s) <= n else s[:n-3] + "..."

err_group = df_error.groupby('cmdb_id').agg(
    count=('message', 'count'),
    min_ts=('ts', 'min'),
    max_ts=('ts', 'max')
).reset_index().sort_values('count', ascending=False)

# Collect up to 5 distinct sample messages per cmdb_id, truncated, join into one string
samples = []
for cmdb in err_group['cmdb_id'].tolist():
    msgs = df_error.loc[df_error['cmdb_id'] == cmdb, 'message'].dropna().astype(str).unique()[:5]
    msgs_trunc = [truncate(m, 200) for m in msgs]
    samples.append(" || ".join(msgs_trunc) if len(msgs_trunc)>0 else "")
err_group['sample_messages'] = samples

# Limit error samples display to top entries by count (top 20 but requirement says up to 5 messages per cmdb_id)
error_summary = err_group.head(20)

# Keep outputs compact: show summary single-row tables and top pairs (up to 20) and error summary (top cmdb_ids)
metric_summary, metric_top_pairs, trace_summary, trace_top_pairs, log_summary, log_top_pairs, error_summary.head(20)

```
Out[1]:
```
summary = (
    "Summary (all times UTC, window 2024-01-23 03:08 → 2024-01-23 03:32):\n\n"
    "metric.csv: 9038 rows. Top (cmdb_id, kpi) pairs each have 25 samples. Notable high P95 values:\n"
    "- ts-admin-basic-info-service: diskio P95 ≈ 5.9e4, mem P95 ≈ 2.32e8, cpu/socket present.\n"
    "- ts-travel2-service: mem P95 ≈ 2.75e8, diskio P95 ≈ 6.16e4, cpu P95 ≈ 11.67, socket P95 ≈ 21.99.\n"
    "- ts-travel-service: diskio P95 ≈ 1.58e5.\n"
    "- ts-voucher-service: socket P95 ≈ 4.0.\n\n"
    "trace.csv: 18392 rows. Top (cmdb_id, trace_name) pairs each have 25 samples. Highlights:\n"
    "- ts-travel2-service shows very large row_count P95 (≈ 2821.8) and many outgoing traces\n"
    "  (to-order, to-route, to-train, to-seat) with non-trivial duration P95s (seat P95 up to ≈0.124s).\n\n"
    "log.csv: 2180 rows. log.row_count is high for several services (P95):\n"
    "- ts-basic-service row_count P95 ≈ 2776.4\n"
    "- ts-travel2-service row_count P95 ≈ 2456.6\n"
    "- ts-travel-service row_count P95 ≈ 2239.8\n"
    "- ts-seat-service row_count P95 ≈ 2066.6\n"
    "log.error_count for top services is 0.0 in the samples.\n\n"
    "error_logs.csv: no entries (empty) — no sampled error messages present.\n\n"
    "Overall impression: the travel-related services (especially ts-travel2-service and ts-travel-service)\n"
    "show high traffic (large row_count in traces and logs) and high resource P95s (disk I/O and memory).\n"
    "No error-log entries were captured in the window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows                    min_ts                    max_ts
0        9038 2024-01-23 03:08:00+00:00 2024-01-23 03:32:00+00:00,                          cmdb_id    kpi_name  count           p95
361           ts-voucher-service      socket     25  4.000000e+00
0    ts-admin-basic-info-service         cpu     25  1.272095e+00
1    ts-admin-basic-info-service      diskio     25  5.915934e+04
2    ts-admin-basic-info-service  latency-50     25  1.840828e-02
3    ts-admin-basic-info-service  latency-90     25  3.473561e-02
4    ts-admin-basic-info-service         mem     25  2.322145e+08
5    ts-admin-basic-info-service      socket     25  7.920000e+00
6    ts-admin-basic-info-service    workload     25  6.716100e-01
7         ts-admin-order-service         cpu     25  1.223488e-01
8         ts-admin-order-service      diskio     25  1.051555e+05
322             ts-travel2-mongo         mem     25  6.905696e+07
323             ts-travel2-mongo      socket     25  8.000000e+00
324           ts-travel2-service         cpu     25  1.166710e+01
325           ts-travel2-service      diskio     25  6.165245e+04
326           ts-travel2-service  latency-50     25  2.116722e-02
327           ts-travel2-service  latency-90     25  4.132816e-01
328           ts-travel2-service         mem     25  2.749483e+08
329           ts-travel2-service      socket     25  2.198636e+01
314            ts-travel-service      diskio     25  1.578717e+05
315            ts-travel-service  latency-50     25  1.936589e-02,    total_rows                    min_ts                    max_ts
0       18392 2024-01-23 03:08:00+00:00 2024-01-23 03:32:00+00:00,                 cmdb_id                                     trace_name  count          p95
7                  root     trace.to_ts-admin-travel-service.row_count     25    10.800000
728  ts-travel2-service  trace.to_ts-order-other-service.duration_mean     25     0.006480
729  ts-travel2-service   trace.to_ts-order-other-service.duration_p95     25     0.012543
730  ts-travel2-service     trace.to_ts-order-other-service.error_rate     25     0.000000
731  ts-travel2-service      trace.to_ts-order-other-service.row_count     25    85.400000
732  ts-travel2-service        trace.to_ts-route-service.duration_mean     25     0.003861
733  ts-travel2-service         trace.to_ts-route-service.duration_p95     25     0.004534
734  ts-travel2-service           trace.to_ts-route-service.error_rate     25     0.000000
748  ts-travel2-service      trace.to_ts-travel2-service.duration_mean     25     0.012693
749  ts-travel2-service       trace.to_ts-travel2-service.duration_p95     25     0.054601
750  ts-travel2-service         trace.to_ts-travel2-service.error_rate     25     0.000000
751  ts-travel2-service          trace.to_ts-travel2-service.row_count     25  2821.800000
736  ts-travel2-service         trace.to_ts-seat-service.duration_mean     25     0.056616
737  ts-travel2-service          trace.to_ts-seat-service.duration_p95     25     0.124000
738  ts-travel2-service            trace.to_ts-seat-service.error_rate     25     0.000000
739  ts-travel2-service             trace.to_ts-seat-service.row_count     25   169.800000
744  ts-travel2-service        trace.to_ts-train-service.duration_mean     25     0.005368
745  ts-travel2-service         trace.to_ts-train-service.duration_p95     25     0.005921
746  ts-travel2-service           trace.to_ts-train-service.error_rate     25     0.000000
747  ts-travel2-service            trace.to_ts-train-service.row_count     25   307.800000,    total_rows                    min_ts                    max_ts
0        2180 2024-01-23 03:08:00+00:00 2024-01-23 03:32:00+00:00,                         cmdb_id         log_name  count     p95
0   ts-admin-basic-info-service  log.error_count     25     0.0
1   ts-admin-basic-info-service    log.row_count     25    30.0
13             ts-basic-service    log.row_count     25  2776.4
12             ts-basic-service  log.error_count     25     0.0
11              ts-auth-service    log.row_count     25   335.2
10              ts-auth-service  log.error_count     25     0.0
85           ts-travel2-service    log.row_count     25  2456.6
84           ts-travel2-service  log.error_count     25     0.0
87              ts-ui-dashboard    log.row_count     25   298.4
86              ts-ui-dashboard  log.error_count     25     0.0
61             ts-price-service    log.row_count     25   310.4
60             ts-price-service  log.error_count     25     0.0
64             ts-route-service  log.error_count     25     0.0
65             ts-route-service    log.row_count     25  1089.4
67              ts-seat-service    log.row_count     25  2066.6
66              ts-seat-service  log.error_count     25     0.0
81            ts-travel-service    log.row_count     25  2239.8
80            ts-travel-service  log.error_count     25     0.0
45       ts-order-other-service    log.row_count     25   513.0
44       ts-order-other-service  log.error_count     25     0.0, Empty DataFrame
Columns: [cmdb_id, count, min_ts, max_ts, sample_messages]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse previously loaded dataframes if present; otherwise load CSVs
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
try:
    df_error
except NameError:
    df_error = pd.read_csv("error_logs.csv")

# Ensure timestamp parsed to UTC datetime column 'ts'
for df in (df_metric, df_trace, df_log, df_error):
    if 'ts' not in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Target services
targets = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Incident window (UTC)
win_start = pd.to_datetime("2024-01-23 03:05:16", utc=True)
win_end   = pd.to_datetime("2024-01-23 03:35:16", utc=True)

# Helper to perform the scan for a dataframe with a name column
def scan_series(df, name_col, target_cmdb_list, top_n=50):
    # Keep only target cmdb_ids
    df_f = df[df['cmdb_id'].isin(target_cmdb_list)].copy()
    if df_f.empty:
        # Return empty dataframe with expected columns
        cols = ['cmdb_id', name_col, 'total_count', 'p95', 'p15',
                'count_in_window', 'count_above_P95', 'earliest_ts_above_P95',
                'count_below_P15', 'earliest_ts_below_P15']
        return pd.DataFrame(columns=cols)
    # Compute global metrics on full series per (cmdb_id, name)
    def pxx(s, q):
        s2 = s.dropna()
        return float(np.nanpercentile(s2.values, q)) if len(s2)>0 else np.nan
    grouped_full = df_f.groupby(['cmdb_id', name_col])['value'].agg(
        total_count='count',
        p95=lambda s: pxx(s,95),
        p15=lambda s: pxx(s,15)
    ).reset_index()
    # Windowed dataframe
    df_w = df_f[(df_f['ts'] >= win_start) & (df_f['ts'] <= win_end)].copy()
    # Merge thresholds into window rows to evaluate comparisons
    if not df_w.empty:
        df_w = df_w.merge(grouped_full, on=['cmdb_id', name_col], how='left')
        # flags
        df_w['above_p95'] = df_w['value'] > df_w['p95']
        df_w['below_p15'] = df_w['value'] < df_w['p15']
        # Aggregations in window
        agg_win = df_w.groupby(['cmdb_id', name_col]).agg(
            count_in_window = ('value','count'),
            count_above_P95 = ('above_p95','sum'),
            earliest_ts_above_P95 = ('ts', lambda s: s[s.index[df_w.loc[s.index,'above_p95']].tolist()].min() if len(s)>0 else pd.NaT),
            count_below_P15 = ('below_p15','sum'),
            earliest_ts_below_P15 = ('ts', lambda s: s[s.index[df_w.loc[s.index,'below_p15']].tolist()].min() if len(s)>0 else pd.NaT)
        ).reset_index()
        # The lambda above is complex; simpler approach: compute earliest timestamps separately
        # Compute earliest timestamps where above_p95 True
        above = df_w[df_w['above_p95']].groupby(['cmdb_id', name_col])['ts'].min().reset_index().rename(columns={'ts':'earliest_ts_above_P95'})
        below = df_w[df_w['below_p15']].groupby(['cmdb_id', name_col])['ts'].min().reset_index().rename(columns={'ts':'earliest_ts_below_P15'})
        agg_win = df_w.groupby(['cmdb_id', name_col]).agg(
            count_in_window = ('value','count'),
            count_above_P95 = ('above_p95','sum'),
            count_below_P15 = ('below_p15','sum')
        ).reset_index().merge(above, on=['cmdb_id', name_col], how='left').merge(below, on=['cmdb_id', name_col], how='left')
    else:
        # No window rows; create empty agg with zero counts
        agg_win = pd.DataFrame(columns=['cmdb_id', name_col, 'count_in_window', 'count_above_P95', 'count_below_P15', 'earliest_ts_above_P95', 'earliest_ts_below_P15'])
    # Combine full-group stats with window stats (left join to include groups with zero in-window)
    res = grouped_full.merge(agg_win, on=['cmdb_id', name_col], how='left')
    # Fill NaNs for counts with zeros
    res['count_in_window'] = res['count_in_window'].fillna(0).astype(int)
    res['count_above_P95'] = res['count_above_P95'].fillna(0).astype(int)
    res['count_below_P15'] = res['count_below_P15'].fillna(0).astype(int)
    # Format earliest timestamps as UTC ISO strings (or empty)
    def fmt_ts(s):
        if pd.isna(s):
            return pd.NA
        # ensure tz-aware UTC
        return pd.to_datetime(s).tz_convert('UTC').strftime('%Y-%m-%dT%H:%M:%SZ')
    res['earliest_ts_above_P95'] = res['earliest_ts_above_P95'].apply(fmt_ts)
    res['earliest_ts_below_P15'] = res['earliest_ts_below_P15'].apply(fmt_ts)
    # Select and order columns as requested
    cols = ['cmdb_id', name_col, 'total_count', 'p95', 'p15',
            'count_in_window', 'count_above_P95', 'earliest_ts_above_P95',
            'count_below_P15', 'earliest_ts_below_P15']
    res = res[cols]
    # Sort by count_in_window desc and limit
    res = res.sort_values('count_in_window', ascending=False).head(top_n).reset_index(drop=True)
    return res

# Perform scans
metric_scan = scan_series(df_metric, 'kpi_name', targets, top_n=50)
trace_scan  = scan_series(df_trace,  'trace_name', targets, top_n=50)
log_scan    = scan_series(df_log,    'log_name', targets, top_n=50)

# Error logs summary
if df_error.empty or df_error[df_error['cmdb_id'].isin(targets)].empty:
    error_summary = pd.DataFrame([{
        'total_rows': len(df_error),
        'min_ts': pd.NaT,
        'max_ts': pd.NaT,
        'note': 'error_logs empty or no entries for target cmdb_ids'
    }])
else:
    df_err_t = df_error[df_error['cmdb_id'].isin(targets)].copy()
    err_grp = df_err_t.groupby('cmdb_id').agg(
        count=('message','count'),
        min_ts=('ts','min'),
        max_ts=('ts','max')
    ).reset_index().sort_values('count', ascending=False)
    # sample up to 5 distinct messages truncated to 200 chars
    def trunc(msg, n=200):
        if pd.isna(msg):
            return msg
        s = str(msg)
        return s if len(s)<=n else s[:n-3]+'...'
    samples = []
    for cm in err_grp['cmdb_id']:
        msgs = df_err_t.loc[df_err_t['cmdb_id']==cm, 'message'].dropna().astype(str).unique()[:5]
        msgs_trunc = [trunc(m,200) for m in msgs]
        samples.append(" || ".join(msgs_trunc))
    err_grp['sample_messages'] = samples
    # format timestamps to ISO UTC
    err_grp['min_ts'] = err_grp['min_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    err_grp['max_ts'] = err_grp['max_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    error_summary = err_grp.rename(columns={'min_ts':'min_ts','max_ts':'max_ts'})

# Display compact results (limited to top 50 as requested)
metric_scan, trace_scan, log_scan, error_summary.head(50)

```
Out[2]:
```
Summary (plain English, all times UTC):

Overall notes
- For the five target services (ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service) every reported metric/trace/log series has 25 total points and all 25 points fall inside the incident window (2024-01-23 03:05:16 → 2024-01-23 03:35:16 UTC). Global P95/P15 were computed over the full series (25 points) before window filtering.
- Error logs contain no entries for the target services (explicitly empty).

1) Metric highlights
- Common patterns: most metric series show 2 samples above the global P95 and 4 samples below the global P15 during the window. Earliest above-P95 timestamps are typically around 03:08–03:25; earliest below-P15 timestamps are often around 03:08–03:14 (exact earliest timestamps are available per series in the table).
- Notable metric P95s (examples from results):
  - ts-auth-service
    - cpu P95 ≈ 38.28 (2 points > P95; earliest above: 2024-01-23T03:08:00Z)
    - diskio P95 ≈ 1.1613e+10 (2 points > P95; earliest above: 2024-01-23T03:25:00Z)
  - ts-travel-service
    - mem P95 ≈ 2.686e+08 (2 points > P95; earliest above: 2024-01-23T03:15:00Z)
    - socket P95 ≈ 17.64 (2 points > P95; earliest above: 2024-01-23T03:08:00Z)
    - workload P95 ≈ 10.61 (2 points > P95; earliest above: 2024-01-23T03:09:00Z)
- Many other KPIs (latency-50/90, diskio, mem, socket) follow the same 2-above-P95 / 4-below-P15 pattern across the services.

2) Trace highlights
- Many trace series show the same count pattern (25 total, 25 in-window; typically 2 > P95 and 4 < P15).
- Examples:
  - ts-auth-service trace.from_root.row_count: P95 = 83.8 (2 above; earliest above 2024-01-23T03:10:00Z).
  - ts-route-service trace.from_ts-travel2-service.row_count: P95 = 625.6 (2 above; earliest above 2024-01-23T03:10:00Z).
  - ts-train-service and ts-route-service have several outgoing trace durations/p95 entries with small absolute latencies but show the same above/below counts.

3) Log highlights
- log.row_count P95s are high for traffic-heavy services:
  - ts-travel-service log.row_count P95 ≈ 2239.8 (2 > P95; earliest above 2024-01-23T03:09:00Z)
  - ts-route-service log.row_count P95 ≈ 1089.4 (2 > P95; earliest above 2024-01-23T03:14:00Z)
  - ts-order-service log.row_count P95 ≈ 571.2 (2 > P95; earliest above 2024-01-23T03:08:00Z)
  - ts-auth-service log.row_count P95 ≈ 335.2
- log.error_count for these services is 0.0 in the sampled data (no error-count spikes).

4) Error logs
- No error_log entries for the target services in the dataset (explicitly empty).

Plain conclusion / recommended focus
- The dataset indicates significant traffic and resource pressure on travel-related services, especially ts-travel-service (very high log.row_count and high memory/disk metrics) and notable resource spikes on ts-auth-service (high CPU and extreme diskio P95). Multiple KPIs across services show a small number of spikes above their P95 within the incident window (2 samples) and a few low samples below P15 (4 samples). Investigate ts-travel-service and ts-auth-service resource usage and high traffic patterns first. Error logs do not provide additional clues.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  total_count           p95           p15  count_in_window  count_above_P95 earliest_ts_above_P95  count_below_P15 earliest_ts_below_P15
0     ts-auth-service         cpu           25  3.828010e+01  1.496840e+01               25                2  2024-01-23T03:08:00Z                4  2024-01-23T03:14:00Z
1     ts-auth-service      diskio           25  1.161265e+10  4.954981e+04               25                2  2024-01-23T03:25:00Z                4  2024-01-23T03:12:00Z
2     ts-auth-service       error           25  0.000000e+00  0.000000e+00               25                0                  <NA>                0                  <NA>
3     ts-auth-service  latency-50           25  1.151486e+00  2.761392e-01               25                2  2024-01-23T03:09:00Z                4  2024-01-23T03:14:00Z
4     ts-auth-service  latency-90           25  2.318997e+00  8.577267e-01               25                2  2024-01-23T03:09:00Z                4  2024-01-23T03:14:00Z
..                ...         ...          ...           ...           ...              ...              ...                   ...              ...                   ...
32  ts-travel-service  latency-50           25  1.936589e-02  1.856282e-02               25                2  2024-01-23T03:15:00Z                4  2024-01-23T03:08:00Z
33  ts-travel-service  latency-90           25  2.860428e-01  1.474831e-01               25                2  2024-01-23T03:17:00Z                4  2024-01-23T03:12:00Z
34  ts-travel-service         mem           25  2.686035e+08  2.660465e+08               25                2  2024-01-23T03:15:00Z                4  2024-01-23T03:08:00Z
35  ts-travel-service      socket           25  1.764000e+01  1.560529e+01               25                2  2024-01-23T03:08:00Z                4  2024-01-23T03:11:00Z
36  ts-travel-service    workload           25  1.060873e+01  7.704193e+00               25                2  2024-01-23T03:09:00Z                4  2024-01-23T03:10:00Z

[37 rows x 10 columns],              cmdb_id                                  trace_name  total_count         p95         p15  count_in_window  count_above_P95 earliest_ts_above_P95  count_below_P15 earliest_ts_below_P15
0    ts-auth-service               trace.from_root.duration_mean           25    1.224548    0.372451               25                2  2024-01-23T03:21:00Z                4  2024-01-23T03:14:00Z
1    ts-auth-service                trace.from_root.duration_p95           25    2.411898    0.833494               25                2  2024-01-23T03:09:00Z                4  2024-01-23T03:14:00Z
2    ts-auth-service                  trace.from_root.error_rate           25    0.000000    0.000000               25                0                  <NA>                0                  <NA>
3    ts-auth-service                   trace.from_root.row_count           25   83.800000   76.600000               25                2  2024-01-23T03:10:00Z                4  2024-01-23T03:08:00Z
4    ts-auth-service    trace.from_ts-auth-service.duration_mean           25    0.016584    0.004324               25                2  2024-01-23T03:08:00Z                4  2024-01-23T03:14:00Z
..               ...                                         ...          ...         ...         ...              ...              ...                   ...              ...                   ...
45  ts-route-service  trace.from_ts-travel2-service.duration_p95           25    0.004534    0.004315               25                2  2024-01-23T03:22:00Z                4  2024-01-23T03:08:00Z
46  ts-route-service    trace.from_ts-travel2-service.error_rate           25    0.000000    0.000000               25                0                  <NA>                0                  <NA>
47  ts-route-service     trace.from_ts-travel2-service.row_count           25  625.600000  538.400000               25                2  2024-01-23T03:10:00Z                4  2024-01-23T03:08:00Z
48  ts-train-service   trace.from_ts-basic-service.duration_mean           25    0.004836    0.004477               25                2  2024-01-23T03:08:00Z                4  2024-01-23T03:17:00Z
49  ts-train-service    trace.from_ts-basic-service.duration_p95           25    0.007333    0.005104               25                2  2024-01-23T03:13:00Z                4  2024-01-23T03:08:00Z

[50 rows x 10 columns],              cmdb_id         log_name  total_count     p95     p15  count_in_window  count_above_P95 earliest_ts_above_P95  count_below_P15 earliest_ts_below_P15
0    ts-auth-service  log.error_count           25     0.0     0.0               25                0                  <NA>                0                  <NA>
1    ts-auth-service    log.row_count           25   335.2   305.0               25                2  2024-01-23T03:10:00Z                4  2024-01-23T03:08:00Z
2   ts-order-service  log.error_count           25     0.0     0.0               25                0                  <NA>                0                  <NA>
3   ts-order-service    log.row_count           25   571.2   281.2               25                2  2024-01-23T03:08:00Z                4  2024-01-23T03:10:00Z
4   ts-route-service  log.error_count           25     0.0     0.0               25                0                  <NA>                0                  <NA>
5   ts-route-service    log.row_count           25  1089.4   897.6               25                2  2024-01-23T03:14:00Z                4  2024-01-23T03:08:00Z
6  ts-travel-service  log.error_count           25     0.0     0.0               25                0                  <NA>                0                  <NA>
7  ts-travel-service    log.row_count           25  2239.8  1222.2               25                2  2024-01-23T03:09:00Z                4  2024-01-23T03:08:00Z,    total_rows min_ts max_ts                                               note
0           0    NaT    NaT  error_logs empty or no entries for target cmdb...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing variables: df_metric, metric_scan
# Ensure required variables exist
try:
    df_metric
    metric_scan
except NameError:
    # load if not present
    df_metric = pd.read_csv("metric.csv")
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
    # compute global thresholds for target services
    targets = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
    df_m_t = df_metric[df_metric['cmdb_id'].isin(targets)].copy()
    def pxx(s,q):
        s2 = s.dropna()
        return float(np.nanpercentile(s2.values, q)) if len(s2)>0 else np.nan
    metric_scan = df_m_t.groupby(['cmdb_id','kpi_name'])['value'].agg(
        total_count='count',
        p95=lambda s: pxx(s,95),
        p15=lambda s: pxx(s,15)
    ).reset_index()

# Targets and window
targets = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
win_start = pd.to_datetime("2024-01-23 03:05:16", utc=True)
win_end   = pd.to_datetime("2024-01-23 03:35:16", utc=True)

# Prepare metric data: ensure ts present
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Filter to targets and window
df_m_w = df_metric[df_metric['cmdb_id'].isin(targets)].copy()
df_m_w = df_m_w[(df_m_w['ts'] >= win_start) & (df_m_w['ts'] <= win_end)].copy()

# Merge global p95/p15 (computed over full series) into windowed rows
thr = metric_scan[['cmdb_id','kpi_name','p95','p15']].copy()
df_m_w = df_m_w.merge(thr, on=['cmdb_id','kpi_name'], how='left')

# Drop rows without thresholds (if any)
df_m_w = df_m_w.dropna(subset=['p95','p15'], how='any').copy()

# Sort for consecutive detection
df_m_w = df_m_w.sort_values(['cmdb_id','kpi_name','ts']).reset_index(drop=True)

# Flags
df_m_w['above_p95'] = df_m_w['value'] > df_m_w['p95']
df_m_w['below_p15'] = df_m_w['value'] < df_m_w['p15']

# Helper to extract sequences for a boolean column
def extract_sequences(df, flag_col, agg_name):
    # df expected sorted by ts
    mask = df[flag_col].astype(bool)
    # Identify consecutive groups (changes in mask)
    grp = (mask != mask.shift()).cumsum()
    df_grp = df.assign(_grp=grp)
    # Only keep True groups
    true_groups = df_grp[mask].groupby(['cmdb_id','kpi_name','_grp'])
    seqs = true_groups.agg(
        seq_len = ('ts','count'),
        seq_start_ts = ('ts','min'),
        seq_end_ts = ('ts','max'),
        max_value = ('value','max'),
        min_value = ('value','min'),
        p95 = ('p95','first'),
        p15 = ('p15','first')
    ).reset_index(drop=False)
    # Keep necessary columns and compute severity depending on agg_name
    if agg_name == 'above':
        seqs = seqs[seqs['seq_len'] >= 2].copy()
        seqs['anomaly_type'] = 'above_p95'
        seqs['metric_value'] = seqs['max_value']
        # severity = max_value / p95 ; handle p95==0 or nan
        seqs['severity'] = seqs.apply(lambda r: np.inf if pd.isna(r['p95']) or r['p95']==0 else r['max_value']/r['p95'], axis=1)
    else:  # below
        seqs = seqs[seqs['seq_len'] >= 2].copy()
        seqs['anomaly_type'] = 'below_p15'
        seqs['metric_value'] = seqs['min_value']
        # severity = p15 / min_value ; handle min_value==0 or nan
        def sev_row(r):
            if pd.isna(r['min_value']) or r['min_value']==0:
                return np.inf
            if pd.isna(r['p15']) or r['p15']==0:
                return np.inf
            return r['p15'] / r['min_value']
        seqs['severity'] = seqs.apply(sev_row, axis=1)
    # Select and format columns
    seqs = seqs[['cmdb_id','kpi_name','anomaly_type','seq_len','seq_start_ts','seq_end_ts','metric_value','severity']]
    # Format timestamps to ISO UTC
    seqs['seq_start_ts'] = pd.to_datetime(seqs['seq_start_ts']).dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    seqs['seq_end_ts'] = pd.to_datetime(seqs['seq_end_ts']).dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    return seqs

# Collect sequences per group
seq_list = []
# Iterate groups to keep memory small
grouped = df_m_w.groupby(['cmdb_id','kpi_name'])
for (cm, kpi), g in grouped:
    # g is sorted by ts due to earlier sort
    # Extract above sequences
    seq_above = extract_sequences(g, 'above_p95', 'above')
    seq_below = extract_sequences(g, 'below_p15', 'below')
    if not seq_above.empty:
        seq_list.append(seq_above)
    if not seq_below.empty:
        seq_list.append(seq_below)

if seq_list:
    anomalies = pd.concat(seq_list, ignore_index=True)
    # Sort by seq_len desc then severity desc
    anomalies = anomalies.sort_values(['seq_len','severity'], ascending=[False, False]).reset_index(drop=True)
    anomalies_top20 = anomalies.head(20).copy()
else:
    anomalies_top20 = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_type','seq_len','seq_start_ts','seq_end_ts','metric_value','severity'])

# Display compact top results
anomalies_top20

```
Out[3]:
```
Plain English summary (consecutive anomaly sequences within 2024-01-23 03:05:16 → 03:35:16 UTC; only sequences with length ≥2 shown):

High-level
- Found 20 top consecutive anomaly sequences (seq_len ≥ 2) across the five services.
- Sequences include both above-P95 spikes and below-P15 dips. Most sequences are short (len 2–4).

Top findings (by seq_len then severity)
1. Three longest sequences (seq_len = 4) are memory dips (below P15):
   - ts-auth-service, kpi mem: 4 points, 2024-01-23T03:13:00Z → 2024-01-23T03:16:00Z, min_value ≈ 2.3626e+08, severity ≈ 1.018.
   - ts-train-service, kpi mem: 4 points, 03:08 → 03:11, min_value ≈ 2.4907e+08, severity ≈ 1.001.
   - ts-travel-service, kpi mem: 4 points, 03:08 → 03:11, min_value ≈ 2.6583e+08, severity ≈ 1.001.
   - These are modest severity (severity ≈1.00–1.02), indicating values slightly below the P15 thresholds.

2. A socket dip on ts-order-service (seq_len = 3):
   - ts-order-service, kpi socket: 3 points, 03:08 → 03:10, min_value ≈ 8.7045, severity ≈ 1.25 (notable relative drop).

3. Several latency and resource dips of length 2–3 across services:
   - latency-50/latency-90 dips on ts-auth, ts-route, ts-travel, ts-order with severity ~1.1–1.66.
   - memory, diskio, cpu dips with seq_len 2 and modest severities (~1.11–1.22).

4. Notable above-P95 spikes (seq_len = 2):
   - ts-order-service, kpi error: 2 points, 03:08 → 03:09, max_value ≈ 0.2879, severity ≈ 7.53 — the largest relative spike found (error rate spike relative to its P95).
   - ts-travel-service, kpi cpu: 2 points, 03:14 → 03:15, max_value ≈ 10.5277, severity ≈ 1.89 — notable CPU spike.
   - ts-travel-service, kpi diskio: 2 points, 03:21 → 03:22, max_value ≈ 1.7566e+05, severity ≈ 1.11.

5. Infinite severity case (needs attention):
   - ts-auth-service, kpi workload: 2 points, 03:08 → 03:09, min_value = 0.0, severity reported as infinite (p15/min_value). This likely indicates a zero reading vs nonzero baseline — investigate data validity or service activity.

Overall interpretation and recommended focus
- The largest relative anomaly is an error-rate spike on ts-order-service (severity ~7.5) — investigate error traces and logs around 03:08–03:09.
- The infinite-severity workload zero on ts-auth-service is suspicious (possible outage/zero reporting) — check service/process and instrumentation at 03:08–03:09.
- ts-travel-service shows short CPU and diskIO spikes (seq_len=2) that are noticeable and coincide with travel-related high traffic seen earlier — investigate resource contention around 03:14–03:22.
- Memory dips for several services exist but with low severity (>P15 by only ~1–2%), likely less urgent.

Actionable next steps
- Inspect traces/logs for ts-order-service at 03:08–03:09 to understand the error spike.
- Verify ts-auth-service workload metric and process health at ~03:08 (zero readings).
- Check resource metrics (CPU, disk I/O) and host-level telemetry for ts-travel-service around 03:14–03:22.
- Correlate with traces and logs (if available) for root-cause confirmation.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name anomaly_type  seq_len          seq_start_ts            seq_end_ts  metric_value  severity
0     ts-auth-service         mem    below_p15        4  2024-01-23T03:13:00Z  2024-01-23T03:16:00Z  2.362552e+08  1.018494
1    ts-train-service         mem    below_p15        4  2024-01-23T03:08:00Z  2024-01-23T03:11:00Z  2.490728e+08  1.000876
2   ts-travel-service         mem    below_p15        4  2024-01-23T03:08:00Z  2024-01-23T03:11:00Z  2.658297e+08  1.000816
3    ts-order-service      socket    below_p15        3  2024-01-23T03:08:00Z  2024-01-23T03:10:00Z  8.704545e+00  1.250688
4     ts-auth-service  latency-50    below_p15        3  2024-01-23T03:14:00Z  2024-01-23T03:16:00Z  2.459436e-01  1.122774
5    ts-route-service         mem    below_p15        3  2024-01-23T03:08:00Z  2024-01-23T03:10:00Z  2.577739e+08  1.000501
6     ts-auth-service    workload    below_p15        2  2024-01-23T03:08:00Z  2024-01-23T03:09:00Z  0.000000e+00       inf
7    ts-order-service       error    above_p95        2  2024-01-23T03:08:00Z  2024-01-23T03:09:00Z  2.879091e-01  7.526379
8   ts-travel-service         cpu    above_p95        2  2024-01-23T03:14:00Z  2024-01-23T03:15:00Z  1.052766e+01  1.894965
9   ts-travel-service  latency-90    below_p15        2  2024-01-23T03:21:00Z  2024-01-23T03:22:00Z  8.886040e-02  1.659716
10    ts-auth-service  latency-90    below_p15        2  2024-01-23T03:18:00Z  2024-01-23T03:19:00Z  5.899851e-01  1.453811
11  ts-travel-service         cpu    below_p15        2  2024-01-23T03:27:00Z  2024-01-23T03:28:00Z  2.495703e+00  1.218894
12  ts-travel-service      diskio    below_p15        2  2024-01-23T03:08:00Z  2024-01-23T03:09:00Z  4.215315e+04  1.191812
13   ts-route-service  latency-50    below_p15        2  2024-01-23T03:08:00Z  2024-01-23T03:09:00Z  3.591028e-03  1.179356
14   ts-train-service         cpu    below_p15        2  2024-01-23T03:27:00Z  2024-01-23T03:28:00Z  2.506468e+00  1.143796
15   ts-route-service  latency-90    below_p15        2  2024-01-23T03:08:00Z  2024-01-23T03:09:00Z  7.946746e-03  1.113944
16   ts-train-service      diskio    below_p15        2  2024-01-23T03:14:00Z  2024-01-23T03:15:00Z  4.453015e+04  1.112819
17  ts-travel-service      diskio    above_p95        2  2024-01-23T03:21:00Z  2024-01-23T03:22:00Z  1.756559e+05  1.112650
18   ts-order-service  latency-90    below_p15        2  2024-01-23T03:14:00Z  2024-01-23T03:15:00Z  1.164202e-02  1.105376
19    ts-auth-service  latency-90    below_p15        2  2024-01-23T03:14:00Z  2024-01-23T03:15:00Z  7.806641e-01  1.098714```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing df_trace and trace_scan if present; otherwise load/compute
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

targets = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Compute global p95/p15 per (cmdb_id, trace_name) if not already available
if not ('trace_scan' in globals() and isinstance(trace_scan, pd.DataFrame) and {'cmdb_id','trace_name','p95','p15'}.issubset(trace_scan.columns)):
    def pct(s, q):
        s2 = s.dropna()
        return float(np.nanpercentile(s2.values, q)) if len(s2)>0 else np.nan
    trace_scan = (
        df_trace[df_trace['cmdb_id'].isin(targets)]
        .groupby(['cmdb_id','trace_name'])['value']
        .agg(total_count='count', p95=lambda s: pct(s,95), p15=lambda s: pct(s,15))
        .reset_index()
    )

# Incident window (UTC)
win_start = pd.to_datetime("2024-01-23 03:05:16", utc=True)
win_end   = pd.to_datetime("2024-01-23 03:35:16", utc=True)

# Filter trace rows to targets and window
df_t_w = df_trace[df_trace['cmdb_id'].isin(targets)].copy()
df_t_w = df_t_w[(df_t_w['ts'] >= win_start) & (df_t_w['ts'] <= win_end)].copy()

# Merge global thresholds (p95/p15) into windowed rows
thr = trace_scan[['cmdb_id','trace_name','p95','p15']].copy()
df_t_w = df_t_w.merge(thr, on=['cmdb_id','trace_name'], how='left')

# Drop rows lacking thresholds
df_t_w = df_t_w.dropna(subset=['p95','p15'], how='any').copy()

# Sort for consecutive detection
df_t_w = df_t_w.sort_values(['cmdb_id','trace_name','ts']).reset_index(drop=True)

# Flags
df_t_w['above_p95'] = df_t_w['value'] > df_t_w['p95']
df_t_w['below_p15'] = df_t_w['value'] < df_t_w['p15']

# Function to extract sequences for a group DataFrame g (sorted by ts)
def extract_group_sequences(g):
    seqs = []
    # handle above_p95
    mask = g['above_p95'].astype(bool)
    if mask.any():
        grp_ids = (mask != mask.shift()).cumsum()
        g2 = g.assign(_grp=grp_ids)
        true_groups = g2[mask].groupby('_grp')
        for gid, sub in true_groups:
            if len(sub) >= 2:
                max_val = sub['value'].max()
                p95 = sub['p95'].iloc[0]
                severity = np.inf if (pd.isna(p95) or p95 == 0) else (max_val / p95)
                seqs.append({
                    'cmdb_id': sub['cmdb_id'].iloc[0],
                    'trace_name': sub['trace_name'].iloc[0],
                    'anomaly_type': 'above_p95',
                    'seq_len': int(len(sub)),
                    'seq_start_ts': pd.to_datetime(sub['ts'].min()).tz_convert('UTC').strftime('%Y-%m-%dT%H:%M:%SZ'),
                    'seq_end_ts': pd.to_datetime(sub['ts'].max()).tz_convert('UTC').strftime('%Y-%m-%dT%H:%M:%SZ'),
                    'extremal_value_in_seq': float(max_val),
                    'severity': severity
                })
    # handle below_p15
    mask2 = g['below_p15'].astype(bool)
    if mask2.any():
        grp_ids = (mask2 != mask2.shift()).cumsum()
        g3 = g.assign(_grp=grp_ids)
        true_groups2 = g3[mask2].groupby('_grp')
        for gid, sub in true_groups2:
            if len(sub) >= 2:
                min_val = sub['value'].min()
                p15 = sub['p15'].iloc[0]
                if pd.isna(min_val) or pd.isna(p15):
                    severity = np.inf
                else:
                    severity = np.inf if min_val == 0 else (p15 / min_val)
                seqs.append({
                    'cmdb_id': sub['cmdb_id'].iloc[0],
                    'trace_name': sub['trace_name'].iloc[0],
                    'anomaly_type': 'below_p15',
                    'seq_len': int(len(sub)),
                    'seq_start_ts': pd.to_datetime(sub['ts'].min()).tz_convert('UTC').strftime('%Y-%m-%dT%H:%M:%SZ'),
                    'seq_end_ts': pd.to_datetime(sub['ts'].max()).tz_convert('UTC').strftime('%Y-%m-%dT%H:%M:%SZ'),
                    'extremal_value_in_seq': float(min_val),
                    'severity': severity
                })
    return seqs

# Iterate groups and collect sequences
all_seqs = []
grouped = df_t_w.groupby(['cmdb_id','trace_name'])
for (cm, tn), g in grouped:
    seqs = extract_group_sequences(g)
    if seqs:
        all_seqs.extend(seqs)

# Build DataFrame
if all_seqs:
    anomalies = pd.DataFrame(all_seqs)
    # Sort by seq_len desc then severity desc (np.inf will sort highest)
    anomalies = anomalies.sort_values(['seq_len','severity'], ascending=[False, False]).reset_index(drop=True)
    # Convert infinite severities to numeric np.inf remain for sorting; later display as 'inf' string
    # Prepare display: convert severity np.inf to string 'inf' for readability while keeping numeric column for sorting
    anomalies_display = anomalies.copy()
    anomalies_display['severity'] = anomalies_display['severity'].apply(lambda x: 'inf' if np.isinf(x) else float(x))
    # Limit to top 20
    anomalies_top20 = anomalies_display.head(20).reset_index(drop=True)
else:
    anomalies_top20 = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_type','seq_len','seq_start_ts','seq_end_ts','extremal_value_in_seq','severity'])

# Output the compact top-20 anomaly sequences
anomalies_top20

```
Out[4]:
```
Summary (plain English, UTC):

What was found
- I scanned traces for the five services during 2024-01-23 03:05:16 → 03:35:16 and found multiple short consecutive anomaly sequences (length ≥2) either well below the global P15 or above the global P95. Results were ranked by sequence length then severity; top 20 sequences are shown in the results.

Top anomalies (high-level)
1. Largest relative drops (below P15, seq_len = 3) — strong reductions in trace durations:
   - ts-order-service, trace.from_root.duration_mean: 3 samples, 03:08:00 → 03:10:00, extremal_value = 0.007375, severity ≈ 16.98 (p15 / min_value).
   - ts-order-service, trace.from_root.duration_p95: 3 samples, same window, extremal_value = 0.008171, severity ≈ 16.73.
   - These indicate trace durations for order.root were much lower than usual (large relative drop) at ~03:08–03:10.

2. Auth-service trace reductions (seq_len = 3):
   - ts-auth-service shows multiple related below-P15 sequences around 03:17:00 → 03:19:00 for trace.from_root.duration_p95, trace.from_ts-auth-service.duration_mean, and to_ts-auth-service.* with severities ~1.23–1.55. These are modest-to-moderate drops in durations.

3. Short above-P95 spikes (seq_len = 2) — elevated durations:
   - ts-auth-service, trace.from_ts-auth-service.duration_mean and to_ts-auth-service.duration_mean: 2 samples, 03:08:00 → 03:09:00, extremal_value = 0.047006, severity ≈ 2.83 (notable spike).
   - ts-order-service, trace.from_ts_travel_service.duration_mean & duration_p95: 2 samples, 03:25:00 → 03:26:00, extremal_values ≈ 0.0123 / 0.0612, severities ≈ 1.21 / 1.24 (moderate spikes).

4. Other items of interest:
   - Several trace.duration_p95 / duration_mean series on order/auth show repeated short below-P15 groups at ~03:08–03:13 with modest severities (~1.04–1.88).
   - ts-route-service shows a below-P15 row_count sequence (2 samples, 03:08 → 03:09) with extremal row_count = 476 and severity ≈ 1.13.

Implications and next steps
- The most extreme relative change is the strong reduction in ts-order-service root durations at ~03:08–03:10 (severity ~17). That large ratio suggests something changed in request handling or sampling that made durations much smaller — investigate traces and request volumes at that time.
- ts-auth-service has both short duration spikes (03:08–03:09) and drops (03:17–03:19) — investigate auth service activity and instrumentation around those times.
- ts-order-service and ts-travel interactions show moderate duration spikes around 03:25–03:26 — check dependency calls between travel → order.
- Correlate these trace anomalies with the metric-level anomalies already observed (CPU/diskio spikes, log.row_count increases) and with any traces/logs/error messages. Note earlier scan showed no error_logs entries for these services in the dataset; use application logs and host metrics to confirm root cause.

If you want, I can next:
- show the exact top-20 sequence rows again, or
- correlate these trace sequences with metric spikes (CPU, diskio, log.row_count) at the same timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                  trace_name anomaly_type  seq_len          seq_start_ts            seq_end_ts  extremal_value_in_seq   severity
0   ts-order-service               trace.from_root.duration_mean    below_p15        3  2024-01-23T03:08:00Z  2024-01-23T03:10:00Z               0.007375  16.983126
1   ts-order-service                trace.from_root.duration_p95    below_p15        3  2024-01-23T03:08:00Z  2024-01-23T03:10:00Z               0.008171  16.732304
2    ts-auth-service                trace.from_root.duration_p95    below_p15        3  2024-01-23T03:17:00Z  2024-01-23T03:19:00Z               0.535834   1.555507
3    ts-auth-service    trace.from_ts-auth-service.duration_mean    below_p15        3  2024-01-23T03:17:00Z  2024-01-23T03:19:00Z               0.002839   1.523130
4    ts-auth-service      trace.to_ts-auth-service.duration_mean    below_p15        3  2024-01-23T03:17:00Z  2024-01-23T03:19:00Z               0.002839   1.523130
5    ts-auth-service               trace.from_root.duration_mean    below_p15        3  2024-01-23T03:17:00Z  2024-01-23T03:19:00Z               0.295944   1.258520
6    ts-auth-service     trace.from_ts-auth-service.duration_p95    below_p15        3  2024-01-23T03:17:00Z  2024-01-23T03:19:00Z               0.002627   1.234887
7    ts-auth-service       trace.to_ts-auth-service.duration_p95    below_p15        3  2024-01-23T03:17:00Z  2024-01-23T03:19:00Z               0.002627   1.234887
8   ts-order-service     trace.from_ts-seat-service.duration_p95    below_p15        3  2024-01-23T03:11:00Z  2024-01-23T03:13:00Z               0.006589   1.113489
9    ts-auth-service    trace.from_ts-auth-service.duration_mean    above_p95        2  2024-01-23T03:08:00Z  2024-01-23T03:09:00Z               0.047006   2.834445
10   ts-auth-service      trace.to_ts-auth-service.duration_mean    above_p95        2  2024-01-23T03:08:00Z  2024-01-23T03:09:00Z               0.047006   2.834445
11  ts-order-service   trace.from_ts-order-service.duration_mean    below_p15        2  2024-01-23T03:08:00Z  2024-01-23T03:09:00Z               0.002346   1.880198
12  ts-order-service     trace.to_ts-order-service.duration_mean    below_p15        2  2024-01-23T03:08:00Z  2024-01-23T03:09:00Z               0.002346   1.880198
13  ts-order-service   trace.from_ts-travel-service.duration_p95    above_p95        2  2024-01-23T03:25:00Z  2024-01-23T03:26:00Z               0.061216   1.243433
14  ts-order-service  trace.from_ts-travel-service.duration_mean    above_p95        2  2024-01-23T03:25:00Z  2024-01-23T03:26:00Z               0.012256   1.212345
15  ts-order-service    trace.from_ts-order-service.duration_p95    below_p15        2  2024-01-23T03:08:00Z  2024-01-23T03:09:00Z               0.003373   1.200291
16  ts-order-service      trace.to_ts-order-service.duration_p95    below_p15        2  2024-01-23T03:08:00Z  2024-01-23T03:09:00Z               0.003373   1.200291
17  ts-route-service     trace.from_ts-travel2-service.row_count    below_p15        2  2024-01-23T03:08:00Z  2024-01-23T03:09:00Z             476.000000   1.131092
18  ts-order-service    trace.from_ts-order-service.duration_p95    below_p15        2  2024-01-23T03:13:00Z  2024-01-23T03:14:00Z               0.003898   1.038554
19  ts-order-service      trace.to_ts-order-service.duration_p95    below_p15        2  2024-01-23T03:13:00Z  2024-01-23T03:14:00Z               0.003898   1.038554```
```

In [ ]:
# In[5]:

import pandas as pd

# reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# ensure ts column with UTC datetimes
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
else:
    # ensure dtype is datetime64[ns, UTC]
    df_trace['ts'] = pd.to_datetime(df_trace['ts'], unit='s', utc=True) if df_trace['ts'].dtype == 'int64' else pd.to_datetime(df_trace['ts'], utc=True)

# Targets and incident window (UTC)
targets = ["ts-order-service","ts-auth-service"]
win_start = pd.to_datetime("2024-01-23 03:05:16", utc=True)
win_end   = pd.to_datetime("2024-01-23 03:35:16", utc=True)

# Filter to targets and window
df_t_window = df_trace[df_trace['cmdb_id'].isin(targets)].copy()
df_t_window = df_t_window[(df_t_window['ts'] >= win_start) & (df_t_window['ts'] <= win_end)].copy()

# A) up to 50 trace rows sorted by timestamp with selected columns and timestamp as UTC ISO
df_a = df_t_window.sort_values('ts')[['ts','cmdb_id','trace_name','value']].copy()
df_a['timestamp'] = df_a['ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
df_a = df_a[['timestamp','cmdb_id','trace_name','value']].head(50).reset_index(drop=True)

# B) aggregate by trace_name: count_in_window, earliest_timestamp, latest_timestamp (UTC ISO), top 50 by count
if df_t_window.empty:
    df_b = pd.DataFrame(columns=['trace_name','count_in_window','earliest_timestamp','latest_timestamp'])
else:
    agg = df_t_window.groupby('trace_name').agg(
        count_in_window = ('value','count'),
        earliest_ts = ('ts','min'),
        latest_ts = ('ts','max')
    ).reset_index()
    agg['earliest_timestamp'] = agg['earliest_ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    agg['latest_timestamp'] = agg['latest_ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    df_b = agg[['trace_name','count_in_window','earliest_timestamp','latest_timestamp']].sort_values('count_in_window', ascending=False).head(50).reset_index(drop=True)

# Display compact outputs
df_a, df_b

```
Out[5]:
```
Summary (plain English, UTC):

A) Raw trace rows (window 2024-01-23 03:05:16 → 03:35:16, services ts-auth-service & ts-order-service)
- Returned up to 50 trace rows sorted by timestamp. Example early rows (03:08:00Z):
  - 2024-01-23T03:08:00Z | ts-auth-service | trace.from_root.duration_mean = 0.448494
  - 2024-01-23T03:08:00Z | ts-order-service | trace.to_ts-order-service.row_count = 198.0
  - 2024-01-23T03:08:00Z | ts-order-service | trace.to_ts-order-service.duration_mean = 0.002346
- The sample covers many per-minute trace entries for both services in the incident window (rows shown span 03:08 → 03:32 with dense activity around 03:08–03:09).

B) Aggregated counts by trace_name (window)
- Top trace_name counts (examples):
  - trace.from_root.duration_mean — count_in_window = 50 (earliest 2024-01-23T03:08:00Z, latest 2024-01-23T03:32:00Z)
  - trace.from_root.duration_p95 — count_in_window = 50 (03:08 → 03:32)
  - trace.from_root.error_rate — count_in_window = 50 (03:08 → 03:32)
  - trace.from_root.row_count — count_in_window = 50 (03:08 → 03:32)
  - trace.from_ts-auth-service.duration_mean — count_in_window = 25 (03:08 → 03:32)
- Many trace names appear 25 times (per-service) or 50 times (present for both services), and some appear ~23 times (less frequent upstream/downstream traces). Earliest timestamps are ~03:08:00Z and latest around ~03:31–03:32:00Z for the most frequent trace_names.

Interpretation / next steps
- The most frequent traces are the root traces (trace.from_root.*) — they show continuous coverage through the window and are highest-volume trace series to inspect for latency or error anomalies.
- For service-specific traces (trace.from_ts-auth-service.*, to_ts-order-service.* etc.) counts reflect service participation (≈25 rows per service). Focus trace.from_root and the top per-service trace_names around 03:08–03:09 and 03:25–03:32 where activity is dense.

The original code execution output of IPython Kernel is also provided below for reference:

(               timestamp           cmdb_id                                trace_name       value
0   2024-01-23T03:08:00Z   ts-auth-service             trace.from_root.duration_mean    0.448494
1   2024-01-23T03:08:00Z  ts-order-service       trace.to_ts-order-service.row_count  198.000000
2   2024-01-23T03:08:00Z  ts-order-service      trace.to_ts-order-service.error_rate    0.000000
3   2024-01-23T03:08:00Z  ts-order-service    trace.to_ts-order-service.duration_p95    0.003664
4   2024-01-23T03:08:00Z  ts-order-service   trace.to_ts-order-service.duration_mean    0.002346
..                   ...               ...                                       ...         ...
45  2024-01-23T03:09:00Z  ts-order-service    trace.from_ts-travel-service.row_count   72.000000
46  2024-01-23T03:09:00Z  ts-order-service   trace.to_ts-order-service.duration_mean    0.002882
47  2024-01-23T03:09:00Z  ts-order-service    trace.to_ts-order-service.duration_p95    0.003373
48  2024-01-23T03:09:00Z  ts-order-service      trace.to_ts-order-service.error_rate    0.000000
49  2024-01-23T03:09:00Z  ts-order-service  trace.to_ts-station-service.duration_p95    0.107149

[50 rows x 4 columns],                                            trace_name  count_in_window    earliest_timestamp      latest_timestamp
0                       trace.from_root.duration_mean               50  2024-01-23T03:08:00Z  2024-01-23T03:32:00Z
1                        trace.from_root.duration_p95               50  2024-01-23T03:08:00Z  2024-01-23T03:32:00Z
2                          trace.from_root.error_rate               50  2024-01-23T03:08:00Z  2024-01-23T03:32:00Z
3                           trace.from_root.row_count               50  2024-01-23T03:08:00Z  2024-01-23T03:32:00Z
4            trace.from_ts-auth-service.duration_mean               25  2024-01-23T03:08:00Z  2024-01-23T03:32:00Z
..                                                ...              ...                   ...                   ...
39  trace.from_ts-inside-payment-service.duration_...               23  2024-01-23T03:09:00Z  2024-01-23T03:31:00Z
40       trace.from_ts-preserve-service.duration_mean               23  2024-01-23T03:09:00Z  2024-01-23T03:31:00Z
41        trace.from_ts-preserve-service.duration_p95               23  2024-01-23T03:09:00Z  2024-01-23T03:31:00Z
42          trace.from_ts-preserve-service.error_rate               23  2024-01-23T03:09:00Z  2024-01-23T03:31:00Z
43           trace.from_ts-preserve-service.row_count               23  2024-01-23T03:09:00Z  2024-01-23T03:31:00Z

[44 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd

# Reuse existing dataframes if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure ts datetime column in UTC for both dataframes
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
else:
    # if ts exists but not datetimelike, convert
    if not pd.api.types.is_datetime64_any_dtype(df_metric['ts']):
        df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

if 'ts' not in df_log.columns:
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
else:
    if not pd.api.types.is_datetime64_any_dtype(df_log['ts']):
        df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Filters
targets = ["ts-order-service","ts-auth-service"]
kpis = ["error","workload","cpu","diskio","mem","socket"]
win_start = pd.to_datetime("2024-01-23 03:05:16", utc=True)
win_end   = pd.to_datetime("2024-01-23 03:35:16", utc=True)

# Metric filter
metric_out = (
    df_metric[
        (df_metric['cmdb_id'].isin(targets)) &
        (df_metric['kpi_name'].isin(kpis)) &
        (df_metric['ts'] >= win_start) &
        (df_metric['ts'] <= win_end)
    ]
    .sort_values('ts')
    .assign(timestamp=lambda d: d['ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ'))
    [['timestamp','cmdb_id','kpi_name','value']]
    .head(50)
    .reset_index(drop=True)
)

# Log filter
log_out = (
    df_log[
        (df_log['cmdb_id'].isin(targets)) &
        (df_log['ts'] >= win_start) &
        (df_log['ts'] <= win_end)
    ]
    .sort_values('ts')
    .assign(timestamp=lambda d: d['ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ'))
    [['timestamp','cmdb_id','log_name','value']]
    .head(50)
    .reset_index(drop=True)
)

metric_out, log_out

```
Out[6]:
```
Summary (plain English, UTC, window 2024-01-23 03:05:16 → 03:35:16):

What the metrics show
- ts-auth-service
  - CPU high at the start of the window: ~39.4% at 2024-01-23T03:08:00Z.
  - Workload and disk I/O present but smaller than order service in absolute terms (examples: workload ~2.76 at 03:11; diskio ~5.4e4 at 03:11).
  - log.row_count is moderate (e.g., 68 at 03:08) and log.error_count is 0.

- ts-order-service
  - High traffic: log.row_count very large early in the window (e.g., 748 at 2024-01-23T03:08:00Z) and remains in the hundreds later (examples shown around 03:19–03:20 with 400+).
  - Workload values around ~4.8–5.5 in the 03:08–03:12 range.
  - Resource signals: memory ~2.58e8 at 03:08, diskio ~4.79e4 at 03:08, socket ~8.70 at 03:08 (socket later ~12.43 at 03:12).
  - log.error_count is 0 in sampled rows.

Key takeaways and next steps
- ts-order-service is handling heavy request volume during the incident window (high log.row_count and non-trivial workload), and shows substantial memory/diskio/socket values — investigate request patterns, upstream traffic sources, and capacity on the order service hosts.
- ts-auth-service shows a notable CPU spike at ~03:08 — inspect auth service CPU usage, threads, and recent deployments or load spikes around that time.
- No log.error_count entries observed in these samples (error_count = 0), so investigate traces and application logs for failures or retries (errors may not be reflected in the aggregated error_count metric).
- Correlate these metric observations with trace anomalies (previous trace scans showed timing anomalies around 03:08–03:10 and 03:25–03:26) to pinpoint dependency issues.

The original code execution output of IPython Kernel is also provided below for reference:

(               timestamp           cmdb_id  kpi_name         value
0   2024-01-23T03:08:00Z   ts-auth-service       cpu  3.935937e+01
1   2024-01-23T03:08:00Z  ts-order-service  workload  4.857614e+00
2   2024-01-23T03:08:00Z  ts-order-service    socket  8.704545e+00
3   2024-01-23T03:08:00Z  ts-order-service       mem  2.583934e+08
4   2024-01-23T03:08:00Z  ts-order-service    diskio  4.794283e+04
..                   ...               ...       ...           ...
45  2024-01-23T03:11:00Z   ts-auth-service  workload  2.760250e+00
46  2024-01-23T03:11:00Z   ts-auth-service    diskio  5.405776e+04
47  2024-01-23T03:11:00Z   ts-auth-service       cpu  1.804010e+01
48  2024-01-23T03:12:00Z  ts-order-service  workload  5.516683e+00
49  2024-01-23T03:12:00Z  ts-order-service    socket  1.243333e+01

[50 rows x 4 columns],                timestamp           cmdb_id         log_name  value
0   2024-01-23T03:08:00Z   ts-auth-service  log.error_count      0
1   2024-01-23T03:08:00Z   ts-auth-service    log.row_count     68
2   2024-01-23T03:08:00Z  ts-order-service  log.error_count      0
3   2024-01-23T03:08:00Z  ts-order-service    log.row_count    748
4   2024-01-23T03:09:00Z   ts-auth-service  log.error_count      0
..                   ...               ...              ...    ...
45  2024-01-23T03:19:00Z   ts-auth-service    log.row_count    336
46  2024-01-23T03:19:00Z  ts-order-service  log.error_count      0
47  2024-01-23T03:19:00Z  ts-order-service    log.row_count    436
48  2024-01-23T03:20:00Z  ts-order-service    log.row_count    442
49  2024-01-23T03:20:00Z  ts-order-service  log.error_count      0

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd

# Reuse existing df_trace if available; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure ts datetime column in UTC
if 'ts' not in df_trace.columns or not pd.api.types.is_datetime64_any_dtype(df_trace['ts']):
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
else:
    # ensure timezone-aware UTC
    df_trace['ts'] = pd.to_datetime(df_trace['ts'], utc=True)

# Candidate services and narrow window (UTC)
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
win_start = pd.to_datetime("2024-01-23 03:07:00", utc=True)
win_end   = pd.to_datetime("2024-01-23 03:11:00", utc=True)

# Filter rows by cmdb_id and time window
df_narrow = df_trace[
    df_trace['cmdb_id'].isin(candidates) &
    (df_trace['ts'] >= win_start) &
    (df_trace['ts'] <= win_end)
].copy()

# Match trace_name patterns: inter-service calls involving order/auth and trace.from_root.*
pattern = r'(to_ts-order-service|to_ts-auth-service|from_ts-order-service|from_ts-auth-service|trace\.from_root)'

df_narrow = df_narrow[df_narrow['trace_name'].str.contains(pattern, regex=True, na=False)].copy()

# Prepare output columns and format timestamp as UTC ISO
df_narrow = df_narrow.sort_values('ts').reset_index(drop=True)
df_narrow['timestamp'] = df_narrow['ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
df_result = df_narrow[['timestamp','cmdb_id','trace_name','value']].head(200)

# Display result (compact table)
df_result

```
Out[7]:
```
Summary (plain English, UTC):

- Data returned: 128 trace rows for the narrow window 2024-01-23 03:07:00 → 03:11:00 across the candidate services.

Key call-direction / ordering observations
- Heavy short-duration inter-service traffic from ts-travel-service to ts-order-service around 03:08→03:11:
  - Repeated trace.to_ts-order-service entries at 03:08:00 (duration_mean ≈ 0.005–0.007 s, row_count ≈ 33) and later at ~03:11 row_count ≈ 43. This indicates travel → order calls are frequent and continuous in the early window.
- Root traces for ts-auth-service and ts-order-service are present at the same times:
  - ts-auth-service trace.from_root.duration_mean ≈ 0.448 s at 03:08:00 (auth root latency noticeably larger than the tiny inter-service durations).
  - ts-order-service trace.from_root.error_rate entries are 0.0 (no obvious error rate in these sampled traces).
- No large error_rate values in these inter-service trace rows (error_rate is 0 in sampled records).
- Ordering: many travel→order calls appear at or just before the earliest anomaly window (03:08), while auth root traces show higher latency at the same timestamps — the timeline suggests travel triggers order activity while auth shows concurrent elevated durations.

Interpretation / next steps
- The earliest anomaly period (~03:08–03:10) shows high volume of travel→order calls (many small-duration calls but sizable row_count) together with higher auth root latency (~0.4–0.45s). Investigate:
  - ts-order-service capacity/queueing (row_count spikes) and whether it is backpressuring callers.
  - ts-auth-service CPU/workload around 03:08 (earlier metric scans showed an auth CPU spike).
  - Correlate these trace sequences with the metric anomalies (CPU/diskio/log.row_count) and full traces to determine whether auth or order contention caused downstream slowdowns.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp            cmdb_id                               trace_name      value
0    2024-01-23T03:08:00Z    ts-auth-service            trace.from_root.duration_mean   0.448494
1    2024-01-23T03:08:00Z  ts-travel-service      trace.to_ts-order-service.row_count  33.000000
2    2024-01-23T03:08:00Z  ts-travel-service     trace.to_ts-order-service.error_rate   0.000000
3    2024-01-23T03:08:00Z  ts-travel-service   trace.to_ts-order-service.duration_p95   0.006874
4    2024-01-23T03:08:00Z  ts-travel-service  trace.to_ts-order-service.duration_mean   0.005363
..                    ...                ...                                      ...        ...
123  2024-01-23T03:11:00Z    ts-auth-service             trace.from_root.duration_p95   0.877841
124  2024-01-23T03:11:00Z    ts-auth-service            trace.from_root.duration_mean   0.427589
125  2024-01-23T03:11:00Z  ts-travel-service     trace.to_ts-order-service.error_rate   0.000000
126  2024-01-23T03:11:00Z   ts-order-service               trace.from_root.error_rate   0.000000
127  2024-01-23T03:11:00Z  ts-travel-service      trace.to_ts-order-service.row_count  43.000000

[128 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```